# 🧠 Mental Health Support Chatbot — Fine-Tuning GPT-Neo 125M

**Model:** EleutherAI/gpt-neo-125m  
**Dataset:** EmpatheticDialogues (Facebook AI Research)  
**Method:** LoRA (Low-Rank Adaptation) via PEFT — trains only 0.5% of parameters

## Instructions
> 1. Go to `Runtime → Change runtime type → T4 GPU`
> 2. Update `HF_USERNAME` in cell 3 with your HuggingFace username
> 3. Run all cells with `Runtime → Run all`
> 4. The final cell will push your model to HuggingFace Hub automatically

## 1. Install Dependencies

In [ ]:
!pip install transformers 'datasets==2.14.7' peft accelerate huggingface-hub -q
print('✅ All dependencies installed!')

In [ ]:
import torch
import json
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from huggingface_hub import login
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration
Paste your HuggingFace token below and update your username.

In [ ]:
import os

# ============================================
# CONFIGURATION — Update these two values!
# ============================================
HF_USERNAME = 'your-hf-username'       # <-- UPDATE THIS with your HuggingFace username
MODEL_REPO_NAME = 'mental-health-gptneo'
BASE_MODEL_ID = 'EleutherAI/gpt-neo-125m'
PUSH_TO_HUB = True

# Training hyperparameters
MAX_LENGTH = 256
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-4
# ============================================

# Load HF token from environment variable (set in Colab secrets or paste below)
HF_TOKEN = os.environ.get('HF_TOKEN', None)
if not HF_TOKEN:
    # Fallback: paste your token here directly
    HF_TOKEN = input('Enter your HuggingFace Token: ')

HUB_MODEL_ID = f'{HF_USERNAME}/{MODEL_REPO_NAME}'

login(token=HF_TOKEN)
print(f'✅ Logged in to HuggingFace Hub')
print(f'Model will be pushed to: {HUB_MODEL_ID}')

## 3. Load and Explore EmpatheticDialogues Dataset
This dataset from Facebook AI Research contains 25,000+ conversation turns labeled with 32 emotions.

In [ ]:
print('Loading EmpatheticDialogues dataset...')
raw_dataset = load_dataset('empathetic_dialogues')

print(f'\nDataset splits:')
for split, data in raw_dataset.items():
    print(f'  {split}: {len(data)} examples')

print(f'\nSample entry:')
sample = raw_dataset['train'][0]
print(f'  Emotion: {sample["emotion"]}')
print(f'  Situation: {sample["situation"]}')
print(f'  Utterance: {sample["utterance"]}')

print(f'\nEmotion distribution (top 10):')
from collections import Counter
emotions = [row['emotion'] for row in raw_dataset['train']]
for emotion, count in Counter(emotions).most_common(10):
    print(f'  {emotion}: {count}')

## 4. Data Preparation

In [ ]:
def format_conversation(example):
    emotion = example['emotion'].replace('_', ' ')
    utterance = example['utterance']
    text = (
        f"[Emotion: {emotion}]\n"
        f"User: {utterance}\n"
        f"Serene: I understand you're feeling {emotion}. "
    )
    return {'text': text}

print('Formatting dataset...')
formatted_train = raw_dataset['train'].map(format_conversation, remove_columns=raw_dataset['train'].column_names, num_proc=2)
formatted_val = raw_dataset['validation'].map(format_conversation, remove_columns=raw_dataset['validation'].column_names, num_proc=2)

print(f'Train size: {len(formatted_train)}')
print(f'Validation size: {len(formatted_val)}')
print(f'\nSample formatted text:\n{formatted_train[0]["text"]}')

In [ ]:
print('Loading tokenizer and tokenizing...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(examples):
    outputs = tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH, padding='max_length')
    outputs['labels'] = outputs['input_ids'].copy()
    return outputs

tokenized_train = formatted_train.map(tokenize, batched=True, remove_columns=['text'], num_proc=2)
tokenized_val = formatted_val.map(tokenize, batched=True, remove_columns=['text'], num_proc=2)
tokenized_train.set_format('torch')
tokenized_val.set_format('torch')

print(f'✅ Tokenization complete!')

## 5. Load Base Model & Apply LoRA

In [ ]:
print('Loading GPT-Neo 125M and applying LoRA...')
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
)

model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied! Trainable: {trainable:,} params ({100*trainable/total:.2f}%)')

## 6. Train the Model

In [ ]:
training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=(device == 'cuda'),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('🚀 Starting training...')
train_result = trainer.train()
print(f'✅ Training complete in {train_result.metrics["train_runtime"]:.0f}s')

## 7. Plot Training Curves

In [ ]:
logs = trainer.state.log_history
train_steps = [x['step'] for x in logs if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in logs if 'loss' in x and 'eval_loss' not in x]
eval_steps = [x['step'] for x in logs if 'eval_loss' in x]
eval_losses = [x['eval_loss'] for x in logs if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, label='Training Loss', color='#6366f1', linewidth=2)
ax.plot(eval_steps, eval_losses, 'o-', label='Validation Loss', color='#f59e0b', linewidth=2, markersize=8)
ax.set_xlabel('Training Step'); ax.set_ylabel('Loss')
ax.set_title('GPT-Neo Fine-Tuning on EmpatheticDialogues', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final train loss: {train_losses[-1]:.4f} | Final val loss: {eval_losses[-1]:.4f}')

## 8. Test the Fine-Tuned Model

In [ ]:
def generate_response(prompt_text, max_new_tokens=100):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(device)
    model.to(device)
    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'], max_new_tokens=max_new_tokens, temperature=0.75,
            top_p=0.92, repetition_penalty=1.3, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).split('User:')[0].strip()

test_cases = [
    ('anxious', "I'm so stressed about my exams. I can't sleep and I keep panicking."),
    ('sad', "I feel like nobody understands me. I'm so alone."),
    ('neutral', "I've been feeling really overwhelmed with work lately."),
]

print('🧪 Testing fine-tuned model responses:\n')
for emotion, user_msg in test_cases:
    prompt = f"[Emotion: {emotion}]\nUser: {user_msg}\nSerene: I understand you're feeling {emotion}."
    response = generate_response(prompt)
    print(f'Emotion: {emotion}')
    print(f'User:    {user_msg}')
    print(f'Serene:  {response}')
    print()

## 9. Push to HuggingFace Hub

In [ ]:
if PUSH_TO_HUB:
    print(f'Pushing to: {HUB_MODEL_ID}')
    merged_model = model.merge_and_unload()
    merged_model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN, commit_message='Fine-tuned GPT-Neo on EmpatheticDialogues')
    tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    print(f'\n✅ Done! View at: https://huggingface.co/{HUB_MODEL_ID}')
    print(f'\n👉 Update HF_USERNAME = "{HF_USERNAME}" in src/config.py')